# Revenue Potential Predictor — EDA

Goal: predict mean daily revenue for any London OA using 5 JENKI stores as training data.

**Features:** TFL station footfall (nearest station), OA population, OA persona proportions (delta vs UK average)

**Target:** mean daily gross revenue (£) per store → mapped to their OA

In [ ]:
import os
import re
import warnings
import subprocess
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from google.cloud import storage

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.figsize'] = (12, 4)

GCS_BUCKET = 'bombe-ml-data'
GCS_PREFIX = 'revenue-predictor/source-data'
LOCAL_PERSONA_FILE = r'C:\Users\ibarn\Downloads\persona_oa_props_impactScore_comm.csv'

# Known JENKI store postcodes — verify/update if needed
STORE_POSTCODES = {
    'borough':       'SE1 9AH',
    'battersea':     'SW11 8EE',
    'canary_wharf':  'E14 5AB',
    'covent_garden': 'WC2H 9NA',
    'spitalfields':  'E1 6EW',
}

print('Libraries loaded OK')

## 1. Load Revel data → mean daily revenue per store

In [ ]:
def gsutil_cat(gcs_path):
    """Read a GCS file as a string via gsutil."""
    result = subprocess.run(
        ['gsutil', 'cat', gcs_path],
        capture_output=True, text=True, encoding='utf-8', errors='replace'
    )
    return result.stdout

def list_gcs(prefix):
    result = subprocess.run(
        ['gsutil', 'ls', f'gs://{GCS_BUCKET}/{prefix}/'],
        capture_output=True, text=True
    )
    return [l.strip() for l in result.stdout.strip().split('\n') if l.strip().endswith('.csv')]


REVEL_DIRS = {
    'borough':       'Revel Data - Borough',
    'battersea':     'Revel Data - Battersea',
    'canary_wharf':  'Revel Data - Canary Wharf',
    'covent_garden': 'Revel Data - Covent Garden',
    'spitalfields':  'Revel Data - Spitalfields',
}

def extract_date_from_filename(path):
    """Extract YYYY-MM-DD from Revel filename (handles both naming conventions)."""
    fname = path.split('/')[-1]
    # New format: ..._{date}.csv  e.g. 2026-01-18
    m = re.search(r'(\d{4}-\d{2}-\d{2})(?:_|.csv)', fname)
    if m:
        return m.group(1)
    return None


def load_store_daily_revenue(store_key, revel_dir):
    prefix = f"{GCS_PREFIX}/{revel_dir}"
    files = list_gcs(prefix)
    print(f"  {store_key}: {len(files)} daily files")

    daily = []
    for f in files:
        date_str = extract_date_from_filename(f)
        if date_str is None:
            continue
        try:
            content = gsutil_cat(f)
            df = pd.read_csv(StringIO(content))
            # Filter to product rows only
            df = df[df['Row Type'] == 'Product']
            total_sales = pd.to_numeric(df['Total Sales'], errors='coerce').sum()
            daily.append({'date': date_str, 'gross_revenue': total_sales})
        except Exception as e:
            pass  # skip malformed files

    df_daily = pd.DataFrame(daily)
    df_daily['date'] = pd.to_datetime(df_daily['date'])
    df_daily = df_daily[df_daily['gross_revenue'] > 0]  # drop zero-revenue days (likely closed)
    return df_daily


print('Loading Revel daily revenue data from GCS...')
store_revenue = {}
for store, rdir in REVEL_DIRS.items():
    store_revenue[store] = load_store_daily_revenue(store, rdir)

print('\nDone.')

In [ ]:
# Compute mean daily revenue per store
mean_revenue = {}
for store, df in store_revenue.items():
    mean_revenue[store] = df['gross_revenue'].mean()
    print(f"{store:15s}  n_days={len(df):4d}  mean_daily_revenue=£{mean_revenue[store]:.2f}  "
          f"  date_range={df['date'].min().date()} → {df['date'].max().date()}")

# Plot daily revenue distributions per store
fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=False)
for ax, (store, df) in zip(axes, store_revenue.items()):
    ax.hist(df['gross_revenue'], bins=20, edgecolor='black')
    ax.axvline(mean_revenue[store], color='red', linestyle='--', label=f'mean=£{mean_revenue[store]:.0f}')
    ax.set_title(store.replace('_', ' ').title())
    ax.set_xlabel('Daily Revenue (£)')
    ax.legend(fontsize=8)
plt.suptitle('Daily Revenue Distributions per Store', y=1.02)
plt.tight_layout()
plt.show()

## 2. Map stores to OA21 codes via postcode lookup

In [ ]:
# Load postcode → OA21 lookup (large file, read relevant columns only)
print('Loading postcode → OA21 lookup from GCS...')
pcd_content = gsutil_cat(f'gs://{GCS_BUCKET}/{GCS_PREFIX}/PCD_OA21_LSOA21_MSOA21_LAD_FEB25_UK_LU.csv')
pcd_df = pd.read_csv(StringIO(pcd_content), usecols=['pcds', 'oa21cd'], low_memory=False)
pcd_df['pcds'] = pcd_df['pcds'].str.strip().str.upper()
print(f'PCD lookup: {len(pcd_df):,} rows')

# Map each store postcode to OA21
store_oa = {}
for store, postcode in STORE_POSTCODES.items():
    match = pcd_df[pcd_df['pcds'] == postcode.upper().strip()]
    if len(match) == 0:
        print(f'  WARNING: no OA found for {store} postcode {postcode}')
        store_oa[store] = None
    else:
        oa = match.iloc[0]['oa21cd']
        store_oa[store] = oa
        print(f'  {store:15s}  {postcode}  →  {oa}')

## 3. TFL station footfall → aggregate to nearest station per OA

In [ ]:
# Load stations_with_oa (gives station lat/lng and OA codes)
print('Loading stations_with_oa...')
stations_oa_content = gsutil_cat(f'gs://{GCS_BUCKET}/{GCS_PREFIX}/stations_with_oa_complete.csv')
stations_oa = pd.read_csv(StringIO(stations_oa_content))

# Get unique station → lat/lng + OA mapping (origin side)
station_meta = stations_oa[[
    'origin_uniquestationname', 'origin_latitude', 'origin_longitude', 'oa21cd_origin'
]].drop_duplicates('origin_uniquestationname').rename(columns={
    'origin_uniquestationname': 'station',
    'origin_latitude': 'lat',
    'origin_longitude': 'lon',
    'oa21cd_origin': 'oa21cd'
})
print(f'Unique stations: {len(station_meta)}')
station_meta.head(3)

In [ ]:
# Load StationFootfall (2024-25 and 2025-26), compute mean annual daily entries per station
print('Loading TFL footfall data...')

ff_2425 = pd.read_csv(StringIO(gsutil_cat(f'gs://{GCS_BUCKET}/{GCS_PREFIX}/StationFootfall_2024_2025 -2.csv')))
ff_2526 = pd.read_csv(StringIO(gsutil_cat(f'gs://{GCS_BUCKET}/{GCS_PREFIX}/StationFootfall_2025_2026.csv')))
footfall_all = pd.concat([ff_2425, ff_2526], ignore_index=True)

print(f'Footfall rows: {len(footfall_all):,}')
print(footfall_all.dtypes)
footfall_all.head(3)

In [ ]:
# Compute mean daily entries per station
footfall_mean = (
    footfall_all
    .groupby('Station')[['EntryTapCount', 'ExitTapCount']]
    .mean()
    .reset_index()
    .rename(columns={'Station': 'station', 'EntryTapCount': 'mean_daily_entries', 'ExitTapCount': 'mean_daily_exits'})
)
footfall_mean['mean_daily_footfall'] = footfall_mean['mean_daily_entries'] + footfall_mean['mean_daily_exits']
print(f'Stations with footfall data: {len(footfall_mean)}')
footfall_mean.describe()

In [ ]:
from math import radians, cos, sin, asin, sqrt

def haversine(lat1, lon1, lat2, lon2):
    """Distance in km between two lat/lon points."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * R * asin(sqrt(a))


def nearest_station_footfall(store_lat, store_lon, station_meta, footfall_mean, top_n=3, radius_km=1.0):
    """Sum footfall from stations within radius_km of a point (or fallback to nearest)."""
    sm = station_meta.copy()
    sm = sm.dropna(subset=['lat', 'lon'])
    sm['dist_km'] = sm.apply(
        lambda r: haversine(store_lat, store_lon, r['lat'], r['lon']), axis=1
    )
    nearby = sm[sm['dist_km'] <= radius_km].sort_values('dist_km')
    if len(nearby) == 0:
        nearby = sm.nsmallest(1, 'dist_km')  # fallback: nearest station

    # Join footfall
    merged = nearby.merge(footfall_mean, on='station', how='left')
    total_footfall = merged['mean_daily_footfall'].sum()
    nearest_name = nearby.iloc[0]['station']
    return total_footfall, nearest_name, nearby['dist_km'].min()


# Get store lat/lng from PCD file (use OA centroid from station_meta as proxy)
# We'll use a simple geocoding: find the station_meta row whose OA matches the store OA
# Fallback: manually hardcoded store lat/lngs
STORE_LATLNG = {
    'borough':       (51.5055, -0.0908),
    'battersea':     (51.4839, -0.1445),
    'canary_wharf':  (51.5054, -0.0235),
    'covent_garden': (51.5129, -0.1243),
    'spitalfields':  (51.5194, -0.0749),
}

store_footfall = {}
for store, (lat, lon) in STORE_LATLNG.items():
    ff, nearest, dist = nearest_station_footfall(lat, lon, station_meta, footfall_mean)
    store_footfall[store] = ff
    print(f"{store:15s}  nearest={nearest:30s}  dist={dist:.2f}km  footfall={ff:,.0f}")

## 4. Population from SAPE (mid-2022 OA estimates)

In [ ]:
import os

SAPE_LOCAL = os.path.join(os.environ['TEMP'], 'sape.xlsx')

if not os.path.exists(SAPE_LOCAL):
    print('Downloading SAPE xlsx from GCS (132 MB)...')
    subprocess.run(
        ['gsutil', 'cp', f'gs://{GCS_BUCKET}/{GCS_PREFIX}/sapeoatablefinal2022v2.xlsx', SAPE_LOCAL],
        check=True
    )

print('Loading SAPE population data...')
sape_raw = pd.read_excel(SAPE_LOCAL, sheet_name='Mid-2022 OA 2021', header=3)
sape = sape_raw[['OA 2021 Code', 'Total']].copy()
sape.columns = ['oa21cd', 'population']
sape = sape.dropna(subset=['oa21cd'])
sape['population'] = pd.to_numeric(sape['population'], errors='coerce')
print(f'SAPE rows: {len(sape):,}')
sape.describe()

In [ ]:
# Population for each store OA
store_population = {}
for store, oa in store_oa.items():
    if oa is None:
        store_population[store] = np.nan
        continue
    match = sape[sape['oa21cd'] == oa]
    pop = match.iloc[0]['population'] if len(match) > 0 else np.nan
    store_population[store] = pop
    print(f"{store:15s}  OA={oa}  population={pop}")

## 5. Persona proportions (delta vs UK average)

In [ ]:
print('Loading persona proportions...')
persona_df = pd.read_csv(LOCAL_PERSONA_FILE)
print(f'Shape: {persona_df.shape}')
persona_df.head(3)

In [ ]:
# Persona columns (all except 'code')
persona_cols = [c for c in persona_df.columns if c != 'code']
print(f'Persona columns ({len(persona_cols)}): {persona_cols}')

# Compute UK average for each persona
uk_avg = persona_df[persona_cols].mean()
print('\nUK average persona proportions:')
print(uk_avg.round(4))

In [ ]:
# Persona delta = store_OA_proportion - UK_average per persona
store_persona_delta = {}
persona_df_indexed = persona_df.set_index('code')

for store, oa in store_oa.items():
    if oa is None or oa not in persona_df_indexed.index:
        print(f'  WARNING: OA {oa} not found in persona data for {store}')
        store_persona_delta[store] = {f'{c}_delta': np.nan for c in persona_cols}
        continue
    oa_props = persona_df_indexed.loc[oa, persona_cols]
    delta = oa_props - uk_avg
    store_persona_delta[store] = {f'{c}_delta': v for c, v in delta.items()}
    print(f"{store:15s}  OA={oa}")
    for col, val in delta.items():
        print(f"    {col}: {val:+.4f}")

## 6. Assemble training dataset (5 rows)

In [ ]:
rows = []
for store in REVEL_DIRS.keys():
    row = {
        'store':              store,
        'oa21cd':             store_oa.get(store),
        'mean_daily_revenue': mean_revenue.get(store, np.nan),
        'footfall':           store_footfall.get(store, np.nan),
        'population':         store_population.get(store, np.nan),
    }
    row.update(store_persona_delta.get(store, {}))
    rows.append(row)

train_df = pd.DataFrame(rows).set_index('store')
print(train_df.to_string())

## 7. EDA — .info(), .describe(), feature distributions

In [ ]:
print('=== .info() ===')
train_df.info()

In [ ]:
print('=== .describe() ===')
train_df.describe().T

In [ ]:
# Distribution plots for each feature (bar chart with 5 points per feature)
feature_cols = [c for c in train_df.columns if c != 'oa21cd']
n = len(feature_cols)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3))
axes = axes.flatten()

stores = train_df.index.tolist()
colors = plt.cm.tab10(np.linspace(0, 1, len(stores)))

for i, col in enumerate(feature_cols):
    ax = axes[i]
    vals = train_df[col].values
    x = np.arange(len(stores))
    bars = ax.bar(x, vals, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace('_', '\n') for s in stores], fontsize=7)
    ax.set_title(col.replace('_', ' '), fontsize=9)
    ax.axhline(0, color='black', linewidth=0.5)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Values per Store (5 Training Points)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of each feature with target (with only 5 points, treat as indicative only)
numeric_cols = train_df.select_dtypes(include=np.number).columns.tolist()
feature_only_cols = [c for c in numeric_cols if c != 'mean_daily_revenue']

corrs = train_df[numeric_cols].corr()['mean_daily_revenue'].drop('mean_daily_revenue').sort_values()

fig, ax = plt.subplots(figsize=(10, max(3, len(corrs) * 0.35)))
colors = ['green' if v > 0 else 'red' for v in corrs]
ax.barh(corrs.index, corrs.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson r with mean_daily_revenue')
ax.set_title('Feature Correlations with Target (n=5, indicative only)')
plt.tight_layout()
plt.show()

print(corrs.round(3))

In [ ]:
# Save assembled training dataset
out_path = os.path.join(os.path.dirname(os.path.abspath('.')), 
                        'experiments', 'revenue_predictor_train.csv')
train_df.reset_index().to_csv('revenue_predictor_train.csv', index=False)
print(f'Saved training dataset (5 rows) to revenue_predictor_train.csv')
train_df